# Hosting Capacity Module - Results Explorer

This notebook demonstrates how to connect to the HCM results database and explore the outputs of a work package using Python.

The HCM service writes results to a PostgreSQL database (or Parquet files) after each run.

## 1. Setup

Standard imports and database configuration. Set `BASE_WP_ID` and `INTERVENTION_WP_ID` to the two work packages you want to compare.

In [22]:
#  CONFIGURE THESE VALUES BEFORE RUNNING

# ID of the base work package (network without intervention)
BASE_WP_ID = "0cb8d9ff-5c36-4fbb-b602-0215de51cdfe"

# ID of the intervention work package (network with intervention applied)
INTERVENTION_WP_ID = "f91ffd5a-7efa-4cf7-ad3a-15582f53611e"  # replace with actual intervention WP

# Feeder to drill into in sections 7 & 8.
# Set to None to auto-select the most constrained feeder.
FEEDER = None

In [23]:
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import sqlalchemy
from sqlalchemy import text

# ── Database connection (credentials loaded from notebook_config.json) ────────
with open("notebook_config.json") as _f:
    _cfg = json.load(_f)
DB_URL = _cfg["db_url"]

# ── Voltage base and limits (volts) ──────────────────────────────────────────
V_BASE = 230
V_MIN  = 0.94 * V_BASE   # 216.2 V
V_MAX  = 1.10 * V_BASE   # 253.0 V

C = {"primary": "#3E5571", "mid": "#5B7FA6", "light": "#7BAFD4", "green": "#3D9970", "amber": "#F4A61D", "red": "#E8524A"}

SOURCE_ORDER = ["Base", "Intervention"]

engine = sqlalchemy.create_engine(DB_URL)
print("Connected.")

Connected.


## 2. Load Results

Load the same tables from both work packages and tag each row with its source (`Base` or `Intervention`).

| Table | What it contains |
|---|---|
| `network_performance_metrics_enhanced` | One row per measurement zone x scenario x season x time-of-day. Filtered to yearly / all to get annual totals. |
| `duration_curves` | Load duration curves (kW vs % of time) per measurement zone. |

In [24]:
def _load_metrics(wp_id, source_label):
    df = pd.read_sql(
        text("""
            SELECT *, (timestamp AT TIME ZONE 'AEST') AS timestamp_local
            FROM public.network_performance_metrics_enhanced
            WHERE work_package_id = :wp_id AND season = 'yearly' AND time_of_day = 'all'
        """),
        engine,
        params={"wp_id": wp_id},
    )
    df["source"] = source_label
    return df

def _load_duration_curves(wp_id, source_label):
    df = pd.read_sql(
        text("SELECT * FROM public.duration_curves WHERE work_package_id = :wp_id AND mz_type = 'FEEDER_HEAD'"),
        engine,
        params={"wp_id": wp_id},
    )
    df["source"] = source_label
    return df

df_raw = pd.concat([
    _load_metrics(BASE_WP_ID, "Base"),
    _load_metrics(INTERVENTION_WP_ID, "Intervention"),
], ignore_index=True)

df_dc = pd.concat([
    _load_duration_curves(BASE_WP_ID, "Base"),
    _load_duration_curves(INTERVENTION_WP_ID, "Intervention"),
], ignore_index=True)

print(f"Performance metrics: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print(f"Duration curves:     {df_dc.shape[0]:,} rows x {df_dc.shape[1]} columns")

Performance metrics: 22 rows x 80 columns
Duration curves:     200 rows x 12 columns


## 3. Preprocess

Filter out non-converging results (extreme voltage or power values) and derive a few helper columns.

In [25]:
df = df_raw.copy()
df["timestamp_local"] = pd.to_datetime(df["timestamp_local"])

# Filter non-converging intervals
df = df.query("peak_import < 100_000_000 and peak_export > -100_000_000")
df = df.query("maximum_section_voltage.between(0.7, 1.3) and minimum_section_voltage.between(0.7, 1.3)")

df["year"] = df["timestamp_local"].dt.year.astype(str)

# Convert voltage columns from per-unit to volts
for _col in ["maximum_section_voltage", "minimum_section_voltage"]:
    df[_col] = df[_col] * V_BASE

df["voltage_spread"] = df["maximum_section_voltage"] - df["minimum_section_voltage"]
df["voltage_status"] = df.apply(
    lambda r: "Overvoltage"  if r["maximum_section_voltage"] > V_MAX else
              "Undervoltage" if r["minimum_section_voltage"] < V_MIN else "Normal",
    axis=1,
)

print(f"Filtered to {df.shape[0]:,} rows.")

Filtered to 22 rows.


## 4. Network overview

A quick summary of what the work package covers and the overall network health.

In [26]:
for source, wp_id in [("Base", BASE_WP_ID), ("Intervention", INTERVENTION_WP_ID)]:
    sub = df[df["source"] == source]
    date_min = sub["timestamp_local"].min().strftime("%b %Y")
    date_max = sub["timestamp_local"].max().strftime("%b %Y")
    print(f"[{source}]  WP: {wp_id}")
    print(f"  Date range : {date_min} – {date_max}")
    print(f"  Feeders    : {sub['feeder'].nunique()}")
    print(f"  Scenarios  : {sub['scenario'].nunique()} ({', '.join(sorted(sub['scenario'].unique()))})")
    print(f"  Years      : {', '.join(sorted(sub['year'].unique()))}")
    print(f"  Intervals  : {sub['interval_count'].sum():,}")
    print()

[Base]  WP: 0cb8d9ff-5c36-4fbb-b602-0215de51cdfe
  Date range : Jan 2030 – Jan 2030
  Feeders    : 1
  Scenarios  : 1 (test_scenario_1)
  Years      : 2030
  Intervals  : 192,720

[Intervention]  WP: f91ffd5a-7efa-4cf7-ad3a-15582f53611e
  Date range : Jan 2030 – Jan 2030
  Feeders    : 1
  Scenarios  : 1 (test_scenario_1)
  Years      : 2030
  Intervals  : 192,720



## 5. Which feeders have the most constraints?

Total constraint hours aggregated per feeder across all scenarios and seasons - the fastest way to identify where the network is under stress.

In [27]:
constraint_cols = {
    "Overvoltage hrs":  "voltage_over_limit_hours",
    "Undervoltage hrs": "voltage_under_limit_hours",
    "Thermal load hrs": "thermal_load_limit_hours",
    "Thermal gen hrs":  "thermal_gen_limit_hours",
}

feeder_agg = df.groupby(["feeder", "source"], as_index=False)[list(constraint_cols.values())].sum()
feeder_agg["total_constraint_hrs"] = feeder_agg[list(constraint_cols.values())].sum(axis=1)

# Rank feeders by base constraint hours so the ordering is stable
top_feeders = (
    feeder_agg[feeder_agg["source"] == "Base"]
    .nlargest(10, "total_constraint_hrs")
    .sort_values("total_constraint_hrs")
)
top_feeder_names = top_feeders["feeder"].tolist()

plot_data = feeder_agg[feeder_agg["feeder"].isin(top_feeder_names)]

fig = px.bar(
    plot_data,
    x="total_constraint_hrs",
    y="feeder",
    color="source",
    orientation="h",
    barmode="group",
    title="Top Feeders by Total Constraint Hours — Base vs Intervention",
    labels={"total_constraint_hrs": "Total constraint hours", "feeder": "", "source": ""},
    color_discrete_map={"Base": C["primary"], "Intervention": C["green"]},
    category_orders={"feeder": top_feeders["feeder"].tolist(), "source": SOURCE_ORDER},
)
fig.update_yaxes(categoryorder="array", categoryarray=top_feeders["feeder"].tolist())
fig.update_layout(legend={"orientation": "h", "y": -0.15}, height=460)
fig.show()

## 6. What type of constraint?

Breaking down by constraint type tells you whether a feeder needs a voltage intervention or a thermal one, or both.

In [28]:
label_map = {v: k for k, v in constraint_cols.items()}

stacked = feeder_agg[feeder_agg["feeder"].isin(top_feeder_names)].melt(
    id_vars=["feeder", "source"],
    value_vars=list(constraint_cols.values()),
    var_name="constraint_type",
    value_name="hours",
)
stacked["constraint_type"] = stacked["constraint_type"].map(label_map)
stacked["feeder"] = pd.Categorical(stacked["feeder"], categories=top_feeders["feeder"].tolist(), ordered=True)
stacked = stacked.sort_values("feeder")

fig = px.bar(
    stacked,
    x="hours",
    y="feeder",
    color="constraint_type",
    facet_col="source",
    orientation="h",
    title="Constraint Type Breakdown — Base vs Intervention",
    color_discrete_map={
        "Overvoltage hrs":  C["red"],
        "Undervoltage hrs": C["amber"],
        "Thermal load hrs": C["mid"],
        "Thermal gen hrs":  C["light"],
    },
    category_orders={"source": SOURCE_ORDER},
    labels={"hours": "Hours", "feeder": "", "constraint_type": ""},
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_yaxes(showticklabels=True)
fig.update_xaxes(matches="x")
fig.update_layout(legend={"orientation": "h", "y": -0.2}, height=460)
fig.show()

## 7. Feeder drill-down

Zooming into a single feeder to see which individual distribution transformers are most constrained. Change `FEEDER` at the top of the notebook to explore a different feeder.

In [29]:
if FEEDER is None:
    FEEDER = top_feeders.iloc[-1]["feeder"]
print(f"Drilling into: {FEEDER}")

feeder_df = df[(df["feeder"] == FEEDER) & (df["mz_type"] == "TRANSFORMER")].copy()
feeder_df["zone_label"] = feeder_df["conducting_equipment_mrid"]

zone_agg = feeder_df.groupby(["zone_label", "source"], as_index=False)[list(constraint_cols.values())].sum()
zone_agg["total_constraint_hrs"] = zone_agg[list(constraint_cols.values())].sum(axis=1)

zone_max = zone_agg.groupby("zone_label")["total_constraint_hrs"].max()
zone_order = zone_max[zone_max > 0].sort_values().index.tolist()
zone_agg = zone_agg[zone_agg["zone_label"].isin(zone_order)]

stacked_zone = zone_agg.melt(
    id_vars=["zone_label", "source"],
    value_vars=list(constraint_cols.values()),
    var_name="constraint_type",
    value_name="hours",
)
stacked_zone["constraint_type"] = stacked_zone["constraint_type"].map(label_map)
stacked_zone["zone_label"] = pd.Categorical(stacked_zone["zone_label"], categories=zone_order, ordered=True)
stacked_zone = stacked_zone.sort_values("zone_label")

n_zones = len(zone_order)
chart_height = max(350, n_zones * 35)

fig = px.bar(
    stacked_zone,
    x="hours",
    y="zone_label",
    color="constraint_type",
    facet_col="source",
    orientation="h",
    title=f"Constraint Hours by Transformer — {FEEDER}",
    color_discrete_map={
        "Overvoltage hrs":  C["red"],
        "Undervoltage hrs": C["amber"],
        "Thermal load hrs": C["mid"],
        "Thermal gen hrs":  C["light"],
    },
    category_orders={"source": SOURCE_ORDER},
    labels={"hours": "Hours", "zone_label": "", "constraint_type": ""},
)
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(matches="x")
fig.update_layout(
    legend={"orientation": "h", "y": -0.3},
    margin={"l": 130},
    height=chart_height,
)
fig.show()

Drilling into: 9049


## 8. Duration curve

The load duration curve shows how often a feeder head operates at each power level. Positive values = net import (load dominant); negative values = net export (generation dominant). The shape reveals how much headroom exists for additional load or generation before the network is stressed.

In [30]:
if FEEDER is None:
    FEEDER = top_feeders.iloc[-1]["feeder"]

dc = df_dc[df_dc["feeder"] == FEEDER].copy()
dc = dc[dc["kw"].abs() < 1e6]

dc_data = dc.groupby(["source", "percentage_of_time"], as_index=False)["kw"].mean()
dc_data = dc_data.sort_values(["source", "percentage_of_time"])

fig = px.line(
    dc_data,
    x="percentage_of_time",
    y="kw",
    color="source",
    color_discrete_map={"Base": C["primary"], "Intervention": C["green"]},
    category_orders={"source": SOURCE_ORDER},
    title=f"Load Duration Curve — {FEEDER} (Feeder Head)",
    labels={"percentage_of_time": "% of time", "kw": "Load (kW)", "source": ""},
)
fig.add_hline(y=0, line_dash="dot", line_color="grey", opacity=0.5)
fig.update_layout(height=420)
fig.show()